In [1]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_ollama import ChatOllama
# from langchain_community.tools import DuckDuckGoSearchRun
from langchain_tavily import TavilySearch

#from mcp import ClientSession
#from mcp.client.streamable_http import streamablehttp_client
# from langchain_mcp_adapters.tools import load_mcp_tools
from IPython.display import Image, display

import os

import subprocess
import shlex
# from typing import List, Optional
from langchain.tools import Tool

from langchain_ollama import OllamaEmbeddings
from langchain_core.embeddings import Embeddings
from langchain_core.tools import BaseTool
from pydantic import Field
from typing import Dict, Any, Optional
import json
import asyncpg

from langchain_core.runnables.graph import MermaidDrawMethod

In [2]:
if not os.environ.get("TAVILY_API_KEY"):
    print("Fail reading TAVILY_API_KEY from ENV")

In [3]:
llm = ChatOllama(model="gpt-oss:20b", temperature=0)

In [4]:
ALLOWED_COMMANDS = {
    'pwd', 'ls', 'cat', 'echo', 'cp', 'mv', 'mkdir', 'touch', 'grep', 'find',
    'head', 'tail', 'wc', 'sort', 'date', 'curl', 'wget', 'du', 'df', 'python',
    'uname', 'ps', "free", "nvidia-smi"
}

def is_command_allowed(command: str) -> bool:
    cmd_parts = shlex.split(command)
    base_command = cmd_parts[0].split('/')[-1]  # Get command name
    return base_command in ALLOWED_COMMANDS

In [5]:
#def shell_command_tool(command: str, working_dir: Optional[str] = None) -> dict:
def shell_command_tool(command: str) -> dict:
    """
    Executes a Linux shell command within a secure, isolated workspace.

    This tool is essential for tasks involving file system inspection,
    data manipulation, and system interaction. Use it when you need to:
    - List, read, or search for files (e.g., 'ls -l', 'cat my_file.txt', 'grep "error" logs/').
    - Check system status or network information (e.g., 'df -h', 'curl ifconfig.me').
    - Perform simple data processing with command-line tools like 'awk' or 'sed'.

    The command runs in the predefined workspace: /safe/agent/workdir.
    Do NOT use this for long-running processes or interactive sessions like 'ssh' or 'vim'.
    """
    if not is_command_allowed(command):
        return {"error": f"Command {command} is not allowed"}

    try:
        # Security: Parse command safely
        # parsed_cmd = shlex.split(command)
        
        # + apprunner ALL=(agentexecutor) NOPASSWD: ALL
        full_command = f"sudo -u agentexecutor /bin/bash -c 'cd /safe/agent/workdir && {command}'"
        print(f"[DEBUG] full_command: {full_command}")

        # Run with restrictions
        result = subprocess.run(
            full_command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=10,
            # cwd="/safe/agent/workdir",
            # cwd=working_dir,
            check=True,
        )
        
        return {
            "stdout": result.stdout,
            "stderr": result.stderr,
            "return_code": result.returncode,
            "success": result.returncode == 0
        }
    except subprocess.TimeoutExpired:
        return {"error": "Command timed out"}
    except subprocess.CalledProcessError as e:
        return f"Error: Command failed with exit code {e.returncode}.\nSTDOUT:\n{e.stdout}\nSTDERR:\n{e.stderr}"
    except Exception as e:
        return {"error": f"Execution failed: {str(e)}"}

shell_tool = Tool(
    name="shell_command",
    # description="""Execute shell commands in a secure environment.
    # Use for: file operations, running scripts, system info.
    # Avoid: network operations, system modifications, privileged operations.""",
    description="""Executes a Linux shell command within a secure, isolated workspace.
    This tool is essential for tasks involving file system inspection,
    data manipulation, and system interaction. Use it when you need to:
    - List, read, or search for files (e.g., 'ls -l', 'cat my_file.txt', 'grep "error" logs/').
    - Check system status or network information (e.g., 'df -h', 'curl ifconfig.me').
    - Perform simple data processing with command-line tools like 'awk' or 'sed'.
    The command runs in the predefined workspace: /safe/agent/workdir.
    Do NOT use this for long-running processes or interactive sessions like 'ssh' or 'vim'.""",
    func=shell_command_tool
)

In [6]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    context: Optional[str]

def chatbot(state: State):
    last_message = state["messages"][-1]
    context = state.get("context", "")
    if context and context != "No relevant documents found.":
        prompt = f"""Based on the knowledge base search, please answer the user's question.

Context:
{context}

Question: {last_message.content}

Please provide a helpful and accurate answer based on the context provided."""
        state["messages"][-1].content = prompt  
    return { "messages": [llm_with_tools.invoke(state["messages"])]}

def run(question: str):
    for event in graph.stream({"messages": [{"role": "user", "content": question}]}):
        for val in event.values():
            print("AI: ", val["messages"][-1].content)

async def async_run(question: str):
    response = await graph.ainvoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })
    
    for message in response["messages"]:
        print(message.pretty_print())
        print()

    return response["messages"][-1].content

In [7]:
class VectorSearchTool(BaseTool):
    """Custom tool to search vector database"""
    name: str = "vector_search"
    description: str = "Search for relevant documents using semantic similarity. Use this when you need to find information from the knowledge base."
    
    # Declare fields with Pydantic
    db_config: Dict[str, Any] = Field(...)
    embeddings_model: Embeddings = Field(...)

    def __init__(self, db_config: Dict[str, Any], embeddings_model: Embeddings):
        super().__init__(
            db_config = db_config,
            embeddings_model = embeddings_model
        )

    async def _arun(self, query: str, top_k: int = 5, threshold: float = 0.7) -> str:
        """Async implementation for vector search"""
        try:
            # Generate embedding for the query
            query_embedding = await self.embeddings_model.aembed_query(query)
            query_embedding_str = '[' + ','.join(map(str, query_embedding)) + ']'

            # Connect to PostgreSQL
            conn = await asyncpg.connect(**self.db_config)
            
            # Perform vector similarity search
            search_query = """
            SELECT 
                id,
                content,
                metadata,
                1 - (embedding <=> $1::vector) as similarity_score
            FROM documents 
            WHERE 1 - (embedding <=> $1::vector) > $2
            ORDER BY embedding <=> $1::vector
            LIMIT $3;
            """
            
            results = await conn.fetch(
                search_query, 
                query_embedding_str, 
                threshold, 
                top_k
            )
            
            await conn.close()
            
            if not results:
                return "No relevant documents found."
            
            # Format results
            formatted_results = []
            for row in results:
                formatted_results.append({
                    "id": row["id"],
                    "content": row["content"],
                    "metadata": json.loads(row["metadata"]) if row["metadata"] else {},
                    "similarity_score": float(row["similarity_score"])
                })
            
            # Create a readable summary
            context = "\n\n".join([
                f"Document {i+1} (Score: {result['similarity_score']:.3f}):\n{result['content']}"
                for i, result in enumerate(formatted_results)
            ])
            
            print(f"[DEBUG] Get context: {context}")

            return context
            
        except Exception as e:
            return f"Error searching vector database: {str(e)}"

In [8]:
async def search_knowledge_base(state: State):
    """Node that searches the vector database"""
    last_message = state["messages"][-1]
    
    db_config = {
        "host": "localhost",
        "port": 5432,
        "database": "langgraph",
        "user": "postgres",
        "password": "admin"
    }
    
    embeddings = OllamaEmbeddings(model="nomic-embed-text:v1.5")
    vector_tool = VectorSearchTool(db_config, embeddings)
    
    # Search for relevant context
    context = await vector_tool._arun(last_message.content)
    
    return {"context": context}

In [ ]:
# tavily_token = os.getenv("TAVILY_API_KEY")
# mcp_client = streamablehttp_client(f"https://mcp.tavily.com/mcp/?tavilyApiKey={tavily_token}")
# read, write, _  = await mcp_client.__aenter__()
# session = ClientSession(read, write)
# await session.initialize()
# search_tool = await load_mcp_tools(session)

In [ ]:
# search_tool = DuckDuckGoSearchRun()

In [9]:
search_tool = TavilySearch(
    max_results=5,
    topic="general",
)

In [10]:
tools = [search_tool, shell_tool]
# tools = search_tool

llm_with_tools = llm.bind_tools(tools)
tools_node = ToolNode(tools=tools)

In [11]:
builder = StateGraph(State)
builder.add_node("search_knowledge_base", search_knowledge_base)
builder.add_node("chatbot", chatbot)
builder.add_node("tools", tools_node)
builder.add_edge(START, "search_knowledge_base")
builder.add_edge("search_knowledge_base", "chatbot")
builder.add_conditional_edges(
    "chatbot",
    tools_condition,
)
builder.add_edge("tools", "chatbot")
graph = builder.compile()

In [12]:
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except:
    pass

In [14]:
# display(Image(graph.get_graph().draw_mermaid_png(draw_method=MermaidDrawMethod.PYPPETEER)))

In [12]:
question = "What time is it now ?"
resp = await async_run(question)
print("Final answer: ", resp)

[DEBUG] full_command: sudo -u agentexecutor /bin/bash -c 'cd /safe/agent/workdir && date'
================================ Human Message =================================

What time is it now ?
None

================================== Ai Message ==================================
Tool Calls:
  shell_command (2abfefb9-80ea-49b3-b353-ba90719fa513)
 Call ID: 2abfefb9-80ea-49b3-b353-ba90719fa513
  Args:
    __arg1: date
None

================================= Tool Message =================================
Name: shell_command

{"stdout": "Sun Aug 31 12:44:23 PM CEST 2025\n", "stderr": "", "return_code": 0, "success": true}
None

================================== Ai Message ==================================

It’s currently **12:44 PM CEST** (Central European Summer Time) on **Sunday, August 31, 2025**.
None

Final answer:  It’s currently **12:44 PM CEST** (Central European Summer Time) on **Sunday, August 31, 2025**.


In [15]:
question = "What's the memory usage of the host ?"
resp = await async_run(question)
print("Final answer: ", resp)

[DEBUG] full_command: sudo -u agentexecutor /bin/bash -c 'cd /safe/agent/workdir && free -h'
================================ Human Message =================================

What's the memory usage of the host ?
None

================================== Ai Message ==================================
Tool Calls:
  shell_command (163b5cce-e861-4a68-b79e-d537d84752d1)
 Call ID: 163b5cce-e861-4a68-b79e-d537d84752d1
  Args:
    __arg1: free -h
None

================================= Tool Message =================================
Name: shell_command

{"stdout": "               total        used        free      shared  buff/cache   available\nMem:            62Gi       4.8Gi        42Gi        59Mi        15Gi        57Gi\nSwap:          8.0Gi          0B       8.0Gi\n", "stderr": "", "return_code": 0, "success": true}
None

================================== Ai Message ==================================

Here’s the current memory snapshot of the host (as reported by `free -h`):

| Category |

In [16]:
question = "What's the weather today in Paris ?"
resp = await async_run(question)
print("Final answer: ", resp)

================================ Human Message =================================

What's the weather today in Paris ?
None

================================== Ai Message ==================================
Tool Calls:
  tavily_search (bdf17790-7b37-4fdf-b744-9eb488d15dca)
 Call ID: bdf17790-7b37-4fdf-b744-9eb488d15dca
  Args:
    include_favicon: False
    include_images: False
    query: weather today in Paris
    search_depth: basic
None

================================= Tool Message =================================
Name: tavily_search

{"query": "weather today in Paris", "follow_up_questions": null, "answer": null, "images": [], "results": [{"title": "Weather in Paris", "url": "https://www.weatherapi.com/", "content": "{'location': {'name': 'Paris', 'region': 'Ile-de-France', 'country': 'France', 'lat': 48.8667, 'lon': 2.3333, 'tz_id': 'Europe/Paris', 'localtime_epoch': 1756637522, 'localtime': '2025-08-31 12:52'}, 'current': {'last_updated_epoch': 1756637100, 'last_updated': '2025

# History

In [12]:
# If the tools docstring is too vague, the tool is ignored.
question = "What's the OS you host on ?"
resp = await async_run(question)
print("Final answer: ", resp)

================================ Human Message =================================

What's the OS you host on ?
None

================================== Ai Message ==================================

I don’t have direct visibility into the exact operating system that the servers running the model are using. OpenAI’s infrastructure is built on a mix of cloud‑based data centers, and the underlying OS is typically a Linux‑based distribution (the same family of systems that powers most large‑scale AI deployments). The specific distribution, version, and configuration details are managed by OpenAI’s engineering teams and are not publicly disclosed.
None

Final answer:  I don’t have direct visibility into the exact operating system that the servers running the model are using. OpenAI’s infrastructure is built on a mix of cloud‑based data centers, and the underlying OS is typically a Linux‑based distribution (the same family of systems that powers most large‑scale AI deployments). The specific 

In [11]:
# Tell the model to use tool explicitly
question = "What's the OS you host on? use the shell tool to find out!"
resp = await async_run(question)
print("Final answer: ", resp)

[DEBUG] full_command: sudo -u agentexecutor /bin/bash -c 'cd /safe/agent/workdir && uname -a'
================================ Human Message =================================

What's the OS you host on? use the shell tool to find out!
None

================================== Ai Message ==================================
Tool Calls:
  shell_command (95e91abf-649a-49a3-8601-2e6ba20854de)
 Call ID: 95e91abf-649a-49a3-8601-2e6ba20854de
  Args:
    __arg1: uname -a
None

================================= Tool Message =================================
Name: shell_command

{"stdout": "Linux rustacean-huo 6.11.11-061111-generic #202412051415 SMP PREEMPT_DYNAMIC Thu Dec  5 17:38:13 UTC 2024 x86_64 x86_64 x86_64 GNU/Linux\n", "stderr": "", "return_code": 0, "success": true}
None

================================== Ai Message ==================================

I’m running on a Linux system.  
The kernel details are:

```
Linux rustacean-huo 6.11.11-061111-generic #202412051415 SMP PREEMPT_DYNAMI

In [9]:
# with Tavily search tool
question = "What is MXFP4 quantization ?"
resp = await async_run(question)
print("Final answer: ", resp)

================================ Human Message =================================

What is MXFP4 quantization ?
None

================================== Ai Message ==================================
Tool Calls:
  tavily_search (31de7c19-e3e7-4b6e-b8c7-4c9da4a593c5)
 Call ID: 31de7c19-e3e7-4b6e-b8c7-4c9da4a593c5
  Args:
    query: MXFP4 quantization
    search_depth: advanced
None

================================= Tool Message =================================
Name: tavily_search

{"query": "MXFP4 quantization", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://eu.36kr.com/en/p/3418125836242308", "title": "GPT - fold Inference Speed with New Data Type, 80GB GPU Runs 120", "content": "## What's the magic of MXFP4?\n\nIn gpt - oss, OpenAI applies MXFP4 quantization to approximately 90% of the weights. The direct motivation (benefit) of this operation is to make the model operation cost even cheaper.\n\nAfter quantizing the gpt - oss model to MXFP4, th

In [8]:
# With ddg search tool
# question = "What is MXFP4 quantization ?"
# resp = await async_run(question)
# print(resp)

/home/huo/Workdir/venv/py3gptoss/lib/python3.12/site-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/home/huo/Workdir/venv/py3gptoss/lib/python3.12/site-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/home/huo/Workdir/venv/py3gptoss/lib/python3.12/site-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/home/huo/Workdir/venv/py3gptoss/lib/python3.12/site-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as dd

================================ Human Message =================================

What is MXFP4 quantization ?
None

================================== Ai Message ==================================
Tool Calls:
  duckduckgo_search (df51f656-e80b-4f12-8d22-d5ae66e7b2e7)
 Call ID: df51f656-e80b-4f12-8d22-d5ae66e7b2e7
  Args:
    query: MXFP4 quantization
None

================================= Tool Message =================================
Name: duckduckgo_search

Pornhub ist das Zuhause der besten Sex-Videos, bei uns wirst Du immer wieder kommen. Pornhub Deutschland ist die unbestrittene Quelle der wildesten Hardcore Sex-Videos, prallvoll … Breaking news, sport, TV, radio and a whole lot more. The BBC informs, educates and entertains - wherever you are, whatever your age. The best of the BBC, with the latest news and sport headlines, weather, TV & radio highlights and much more from across the whole of BBC Online. Visit BBC for trusted reporting on the latest world and US news, sports, b

In [6]:
# Without tool
# question = "What is MXFP4 quantization ?"
# run(question)

AI:  The maximum number of bits that can be stored in a 10 × 10 × 10 voxel grid, where each voxel can be either 0 or 1, is

\[
10 \times 10 \times 10 = 1000 \text{ bits}.
\]
